<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_6/bonus/cv_maker/note_bonus_cv_maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Бонус М6 — CV розробника: HTML → PDF

CV — теж маленький інженерний проєкт: **дані** (що ти зробив) → **шаблон** (як це виглядає) → **PDF**. А правила сильного CV з туторіалу (`CV_mini_tutorial_UA.md`) — це перевірки, які можна написати кодом: дієслово дії, лише справжні числа, карта навичок, ключові слова вакансії, відсутність заготовок. Наскрізний приклад — CV **вигаданої** студентки курсу Іванки Прикладенко (`example.com`, `example-student`): підстав свої дані, коли робитимеш своє.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | підготовка: папка `cv_maker` і факти про проєкти курсу | 0.1 | |
| 1 | **сильний пункт**: формула, перевірка, переписування, Professional Summary | 1.1–1.4 | |
| 2 | **чесні числа**: факти з коду, а не з пам'яті | 2.1 | **Задача 1** |
| 3 | **навички й ATS**: що бачить робот, карта компетенцій, ключові слова вакансії | 3.1–3.2 | **Задача 2** |
| 4 | **CV як дані → HTML**: знайди помилку з `<` і `&`, екранування, сторінка зі стилем шаблону, дві версії | 4.1–4.4 | **Задача 3** |
| 5 | **PDF**: який генератор є, PDF або HTML, CSS для друку | 5.1–5.3 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- PDF потребує Chromium (Playwright) або `wkhtmltopdf`. Якщо їх немає (як у Colab «з коробки»), ноутбук показує готовий HTML — його можна відкрити в браузері й надрукувати в PDF (`Ctrl+P`).
- Туторіал про формулювання і розбір обох генераторів — у книзі: [Бонус. CV розробника](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/bonus_cv/).

---
## 0. Підготовка

### Приклад 0.1. Папка проєкту і факти

`find_course_root` шукає репозиторій курсу вгору від поточної папки, а в Colab клонує лише потрібне: `cv_maker`, проєкти уроку 51 і workflows CI (з них у розділі 2 візьмемо справжні числа). Згенеровані CV — у папці `cv_output` поруч з ноутбуком.

In [ ]:
import importlib.util
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
LESSON = "module_6/bonus/cv_maker"
PROJECTS = "module_5/lessons/lesson_51_ci_cd"


def find_course_root(marker: str, sparse: list[str]) -> Path:
    """Корінь репозиторію курсу: шукаємо вгору від поточної папки; не знайшли (Colab) — клонуємо лише потрібні папки."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / marker).is_dir():
            return folder
    target = here / "course_repo"
    if not target.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, str(target)],
                       check=True)
        subprocess.run(["git", "-C", str(target), "sparse-checkout", "set", *sparse], check=True)
    return target


# Факти про проєкти курсу — лише те, що справді є в коді й перевірено запуском (уроки 37–51)
FACTS = {
    "news_hub": {"tests": 343, "ci_jobs": 6, "python_versions": 2, "image_mb": 415, "news": 168,
                 "search_ms_before_index": 372},          # EXPLAIN ANALYZE пошуку, урок 52
    "crispy_notes": {"tests": 81, "replicas": 2},
}

In [ ]:
if importlib.util.find_spec("yaml") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

ROOT = find_course_root(LESSON, [LESSON, PROJECTS, ".github/workflows"])
HERE = ROOT / LESSON
OUT = Path("cv_output")
OUT.mkdir(exist_ok=True)
print(sorted(p.name for p in HERE.iterdir() if not p.name.startswith(".")))

---
## 1. Сильний пункт CV

З туторіалу (розділ 2):

```text
Сильне дієслово + що зроблено + чим/на яких даних + для чого/яка цінність
```

Слабкий пункт описує **процес** («Responsible for…», «Worked on…»), сильний — **результат**.

### Приклад 1.1. Пункт за формулою

Задача: не писати пункти «з голови». Невелика зміна — чотири частини формули як окремі поля, а пункт складає функція. Так видно, якої частини бракує.

In [ ]:
def bullet(verb: str, what: str, tools: list[str], value: str) -> str:
    """Пункт CV за формулою: дієслово + що + чим + яка цінність."""
    return f"{verb} {what} with {', '.join(tools[:-1])} and {tools[-1]}, {value}."


PARTS = [
    ("Built", "a news aggregator API", ["FastAPI", "PostgreSQL", "Redis"], "covered by 343 pytest tests"),
    ("Containerized", "two web applications", ["Docker", "Compose", "nginx"], "started with one command"),
    ("Automated", "pull request checks", ["GitHub Actions", "pytest", "mypy"], "6 jobs on every pull request"),
]

In [ ]:
for parts in PARTS:
    text = bullet(*parts)
    print(f"{len(text.split()):>2} слів: {text}")

Кожен пункт — до 15 слів, з дієсловом, інструментами й результатом. Числа (343, 6) — з `FACTS`, тобто з реальних запусків курсу.

### Приклад 1.2. Перевірка пункту

Задача: перевірити вже написане. `weak_reasons(bullet)` повертає причини, чому пункт слабкий (порожній список — сильний):

- `"слабкий початок"` — починається з `Responsible for`, `Worked on`, `Helped`, `Participated in`, `Was involved in`, `Did`;
- `"немає дієслова дії"` — перше слово не з `STRONG_VERBS`;
- `"задовгий"` — понад 30 слів;
- `"немає інструментів"` — не згадано жодної технології з `TOOLS`.

In [ ]:
STRONG_VERBS = {"Built", "Designed", "Developed", "Automated", "Implemented", "Deployed", "Reduced", "Improved",
                "Migrated", "Containerized", "Tested", "Secured", "Integrated", "Optimized", "Created", "Led"}
WEAK_STARTS = ("responsible for", "worked on", "helped", "participated in", "was involved in", "did")
TOOLS = {"Python", "FastAPI", "Django", "PostgreSQL", "Redis", "Docker", "Compose", "nginx", "GitHub Actions",
         "pytest", "mypy", "SQLAlchemy", "Alembic", "aiogram", "WebSocket", "Channels", "Pydantic", "JWT"}


def weak_reasons(bullet: str) -> list[str]:
    """Причини, чому пункт CV слабкий; [] — сильний."""
    reasons = []
    text = bullet.strip()
    if text.lower().startswith(WEAK_STARTS):
        reasons.append("слабкий початок")
    if text.split()[0] not in STRONG_VERBS:
        reasons.append("немає дієслова дії")
    if len(text.split()) > 30:
        reasons.append("задовгий")
    if not any(tool in text for tool in TOOLS):
        reasons.append("немає інструментів")
    return reasons


SAMPLES = [
    "Responsible for backend development.",
    "Worked on a news project with FastAPI.",
    "Built a news aggregator API with FastAPI, PostgreSQL and Redis, covered by 343 pytest tests.",
    "Containerized two web applications with Docker and Compose behind nginx.",
]

In [ ]:
for text in SAMPLES:
    print(f"{str(weak_reasons(text)):<60} {text}")
assert weak_reasons(SAMPLES[0]) == ["слабкий початок", "немає дієслова дії", "немає інструментів"]
assert weak_reasons(SAMPLES[1]) == ["слабкий початок", "немає дієслова дії"]
assert weak_reasons(SAMPLES[2]) == [] and weak_reasons(SAMPLES[3]) == []
assert all(weak_reasons(bullet(*parts)) == [] for parts in PARTS)

«Worked on … with FastAPI» інструмент має, але це опис процесу: що саме зроблено і з яким результатом — невідомо. Пункти з прикладу 1.1 проходять перевірку всі.

### Приклад 1.3. Переписати слабкі пункти про проєкти курсу

Задача: три типові чернетки — переписати так, щоб `weak_reasons` повернув `[]`, **і** щоб кожен пункт відповідав фактам курсу (нічого не вигадуючи).

In [ ]:
WEAK = [
    "Worked on a Telegram bot.",
    "Helped with Docker stuff for the project.",
    "Did tests and CI.",
]
STRONG = [
    "Built a Telegram bot with aiogram for news digests and keyword subscriptions, sharing the FastAPI app's repository layer.",
    "Containerized a FastAPI news aggregator with Docker and Compose: nginx, PostgreSQL, Redis and one-shot Alembic migrations.",
    "Automated testing with GitHub Actions: 6 jobs per pull request, including pytest on 2 Python versions and mypy --strict.",
]

In [ ]:
for before, after in zip(WEAK, STRONG):
    print(f"✗ {before:<44} {weak_reasons(before)}")
    print(f"✓ {after}\n")
assert all(weak_reasons(text) == [] for text in STRONG)

Кожен сильний пункт можна пояснити на співбесіді: бот — урок 48, Compose з разовими міграціями — урок 50, шість jobs CI на двох версіях Python — урок 51.

### Приклад 1.4. Professional Summary — elevator pitch, а не біографія

Туторіал (розділ 1): Summary = **роль + досвід/бекграунд + спеціалізація + доказ + ключові технології**. Задача: зібрати Summary з частин і перевірити, що жодної не бракує і текст не довший за 60 слів.

In [ ]:
SUMMARY_PARTS = ("роль", "бекграунд", "спеціалізація", "доказ", "технології")


def summary(parts: dict[str, str]) -> tuple[str, list[str]]:
    """(текст Summary, список проблем) — бракує частин або понад 60 слів."""
    missing = [name for name in SUMMARY_PARTS if not parts.get(name)]
    text = (f"{parts.get('роль', '')} with {parts.get('бекграунд', '')}, focused on {parts.get('спеціалізація', '')}. "
            f"{parts.get('доказ', '')} Stack: {parts.get('технології', '')}.")
    issues = [f"бракує: {name}" for name in missing]
    if len(text.split()) > 60:
        issues.append(f"{len(text.split())} слів — скороти до 60")
    return text, issues


WEAK_SUMMARY = {"роль": "Python Developer, Data Analyst, Backend Engineer, DevOps, ML Engineer"}
STRONG_SUMMARY = {
    "роль": "Junior Python Backend Developer",
    "бекграунд": "a background in logistics operations",
    "спеціалізація": "APIs, databases and deployment",
    "доказ": "Built and deployed a FastAPI news service with 343 tests and CI on every pull request.",
    "технології": "FastAPI, Django, PostgreSQL, Redis, Docker, GitHub Actions",
}

In [ ]:
for name, parts in (("слабкий", WEAK_SUMMARY), ("сильний", STRONG_SUMMARY)):
    text, issues = summary(parts)
    print(f"{name}: {issues or '✓'}\n  {text}\n")

Слабкий варіант — перелік ролей: рекрутер не розуміє, хто ти. Сильний за 30 слів відповідає на «хто, звідки, що вмієш, чим доведено». Бекграунд «logistics operations» — частина вигаданої історії Іванки; свій чесний бекграунд (попередня професія, навчання) — перевага, а не те, що треба ховати.

---
## 2. Лише чесні числа

Туторіал (розділ 4): метрики — **тільки справжні**. Число в CV — обіцянка, що ти зможеш його пояснити на співбесіді.

### Приклад 2.1. Факти з коду, а не з пам'яті

Задача: звідки взялися числа у `FACTS`? Невелика зміна — порахуємо їх з репозиторію: тестові функції через `ast`, jobs і версії Python — з YAML workflow CI.

In [ ]:
import ast

import yaml


def test_functions(folder: Path) -> tuple[int, int]:
    """(функцій test_*, з них з @pytest.mark.parametrize) у файлах test*.py папки."""
    total = parametrized = 0
    for path in folder.rglob("test*.py"):
        for node in ast.walk(ast.parse(path.read_text(encoding="utf-8"))):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test"):
                total += 1
                parametrized += any("parametrize" in ast.unparse(d) for d in node.decorator_list)
    return total, parametrized


def ci_facts(workflow: Path) -> dict[str, object]:
    """Jobs і версії Python з матриці workflow GitHub Actions."""
    spec = yaml.safe_load(workflow.read_text(encoding="utf-8"))
    versions = spec["jobs"]["tests"].get("strategy", {}).get("matrix", {}).get("python", [])
    return {"jobs": list(spec["jobs"]), "python": versions}

In [ ]:
news_tests = test_functions(ROOT / PROJECTS / "news_hub" / "tests")
notes_tests = test_functions(ROOT / PROJECTS / "crispy_notes_project")
print(f"news_hub:     {news_tests[0]} функцій test_*, з них {news_tests[1]} з parametrize")
print(f"crispy_notes: {notes_tests[0]} функцій test_*, з них {notes_tests[1]} з parametrize")
for name in ("news_hub", "crispy_notes"):
    facts = ci_facts(ROOT / ".github" / "workflows" / f"{name}.yml")
    print(f"CI {name:<13} {len(facts['jobs'])} jobs {facts['jobs']}, Python {facts['python']}")

Числа CI збігаються з `FACTS`: 6 jobs і 2 версії Python у `news_hub`. 81 тест нотаток — рівно 81 функція. А в `news_hub` функцій 202, тоді як `pytest` у CI звітує `343 passed`: кожна функція з `parametrize` — це кілька тестів (по одному на набір параметрів). Для CV правильне число — те, що **друкує pytest** (урок 50: `343 passed`), бо саме його покаже будь-хто, хто запустить твої тести. Число «з пам'яті» тут легко помилитися.

### Задача 1. Числа без підтвердження

Напиши `unverified_numbers(bullet, facts)` → список **цілих** чисел з пункту (у порядку появи), яких немає серед значень `facts` (усіх проєктів). Версії на кшталт `3.13` не рахуй: число, до якого прилягає крапка з цифрою (`3.13`, `2.0`), пропускай. Підказка: `re.findall(r"(?<![\d.])\d+(?![\d.])", text)`.

🔮 **Прогноз:** які числа в `CLAIMS` не пройдуть перевірку?

In [ ]:
CLAIMS = [
    "Built a news aggregator API with FastAPI, covered by 343 pytest tests.",
    "Improved API performance by 70% with Redis caching.",
    "Automated CI with GitHub Actions: 6 jobs per pull request on 2 Python versions.",
    "Deployed a Django chat app to 2 replicas serving 10000 users.",
    "Migrated the project to Python 3.13 and SQLAlchemy 2.0.",
]

In [ ]:
def unverified_numbers(bullet: str, facts: dict) -> list[int]:
    # BEGIN SOLUTION
    known = {value for project in facts.values() for value in project.values()}
    numbers = [int(n) for n in re.findall(r"(?<![\d.])\d+(?![\d.])", bullet)]
    return [n for n in numbers if n not in known]
    # END SOLUTION

In [ ]:
for claim in CLAIMS:
    print(f"{str(unverified_numbers(claim, FACTS)):<10} {claim}")
assert isinstance(unverified_numbers(CLAIMS[0], FACTS), list), "поверни список чисел"
assert unverified_numbers(CLAIMS[1], FACTS) == [70], "70 — немає серед фактів"
assert [unverified_numbers(c, FACTS) for c in CLAIMS] == [[], [70], [], [10000], []], "версії 3.13 і 2.0 — не метрики, їх пропускаємо"
print("✅ Задача 1 розв'язана")

<details>
<summary>Чому 70% і 10000 — погано</summary>

Кеш Redis у `news_hub` справді є (урок 40), але приріст швидкості ніхто не міряв, а користувачів у навчального проєкту немає. Перше ж питання «як ви це виміряли?» — і довіри до решти CV теж немає. Чесні альтернативи: «with Redis cache-aside and versioned invalidation» (що зроблено) або число, яке ти справді виміряв сам (як `EXPLAIN ANALYZE` в уроці 52: 372 мс → 0,14 мс).
</details>

---
## 3. Навички і ATS

ATS (applicant tracking system) — програма, через яку рекрутер шукає кандидатів. Вона читає **текст** CV: картинки, колонки й текст у графіці вона бачить погано або не бачить зовсім. Туторіал (розділ 9): простий дизайн читається краще.

### Приклад 3.1. Що бачить робот

Задача: подивитися на `cv_template.html` очима ATS — лише текст, у порядку появи, без стилів. `HTMLParser` зі стандартної бібліотеки збирає текстові вузли; вміст `<style>` пропускаємо.

In [ ]:
from html.parser import HTMLParser


class TextOnly(HTMLParser):
    """Текст сторінки рядками — приблизно те, що читає ATS."""

    def __init__(self) -> None:
        super().__init__()
        self.lines: list[str] = []
        self.skip = False

    def handle_starttag(self, tag, attrs):
        self.skip = tag in ("style", "script", "title")

    def handle_endtag(self, tag):
        self.skip = False

    def handle_data(self, data):
        if not self.skip and data.strip():
            self.lines.append(" ".join(data.split()))


def ats_text(html_text: str) -> list[str]:
    parser = TextOnly()
    parser.feed(html_text)
    return parser.lines

In [ ]:
TEMPLATE_HTML = (HERE / "cv_template.html").read_text(encoding="utf-8")
lines = ats_text(TEMPLATE_HTML)
print(len(lines), "рядків тексту; перші вісім:")
for line in lines[:8]:
    print("  ", line)
print("заготовок [..] у шаблоні:", len(re.findall(r"\[[^\]]+\]", TEMPLATE_HTML)))

Порядок тексту — той самий, що на сторінці: ім'я, роль, контакти, Summary, навички. Шаблон простий (без колонок і картинок), тож ATS прочитає все. А ось `[заготовки]` ATS теж прочитає — тому їх не має лишитись жодної (задача 3).

### Задача 2. Карта компетенцій замість списку

Туторіал (розділ 6): список з 20 технологій через кому — слабко; **карта компетенцій** (Backend, Data & storage, Infrastructure, Quality) — сильно. Напиши `skills_map(skills)` → словник `{група: [навички в порядку появи]}` за словником `SKILL_GROUPS`; навичка, якої немає в `SKILL_GROUPS`, — у групу `"Other"`. Груп без навичок у результаті немає; порядок груп — як у `GROUP_ORDER` (і `"Other"` — останньою).

In [ ]:
GROUP_ORDER = ["Backend", "Data & storage", "Infrastructure", "Quality"]
SKILL_GROUPS = {
    "Python": "Backend", "FastAPI": "Backend", "Django": "Backend", "DRF": "Backend", "SQLAlchemy": "Backend",
    "Pydantic": "Backend", "aiogram": "Backend",
    "PostgreSQL": "Data & storage", "Redis": "Data & storage", "Alembic": "Data & storage", "pandas": "Data & storage",
    "Docker": "Infrastructure", "Compose": "Infrastructure", "nginx": "Infrastructure", "GitHub Actions": "Infrastructure",
    "Linux": "Infrastructure",
    "pytest": "Quality", "mypy": "Quality",
}
FLAT = ["Python", "Docker", "pytest", "PostgreSQL", "FastAPI", "Marp", "Redis", "GitHub Actions", "Django", "mypy", "nginx"]

In [ ]:
def skills_map(skills: list[str]) -> dict[str, list[str]]:
    # BEGIN SOLUTION
    groups: dict[str, list[str]] = {}
    for skill in skills:
        groups.setdefault(SKILL_GROUPS.get(skill, "Other"), []).append(skill)
    order = GROUP_ORDER + ["Other"]
    return {name: groups[name] for name in order if name in groups}
    # END SOLUTION

In [ ]:
grouped = skills_map(FLAT)
assert isinstance(grouped, dict), "поверни словник {група: [навички]}"
for group, items in grouped.items():
    print(f"{group + ':':<16} {', '.join(items)}")
assert grouped["Backend"] == ["Python", "FastAPI", "Django"], "навички в групі — у порядку появи у списку"
assert list(grouped) == ["Backend", "Data & storage", "Infrastructure", "Quality", "Other"], "порядок груп — GROUP_ORDER, Other останньою"
assert grouped["Other"] == ["Marp"] and "Other" not in skills_map(["Python"])
print("✅ Задача 2 розв'язана")

Тепер рекрутер за дві секунди бачить профіль: бекенд з базами, деплоєм і тестами. `Marp` (інструмент для слайдів з уроку 53) у «Other» — сигнал подумати, чи він взагалі потрібен у CV бекенд-розробника.

### Приклад 3.2. Ключові слова вакансії

Туторіал (розділ 10): **дзеркаль** ключові слова вакансії — якщо ти справді маєш навичку, назви її так само, як у вакансії. Задача: знайти, які технології з вакансії є в CV, а яких бракує. Невелика зміна до 3.1 — порівнюємо текст вакансії і текст CV за тим самим словником технологій (`SKILL_GROUPS`), без урахування регістру.

In [ ]:
VACANCY = """Junior Python Developer. Requirements: Python, FastAPI or Django, PostgreSQL, Docker.
Nice to have: Redis, Celery, Kubernetes, CI with GitHub Actions, pytest."""
CV_TEXT = " ".join(STRONG + ["Python, FastAPI, Django, PostgreSQL, Redis, Docker, Compose, nginx"])


def keywords(text: str, vocabulary: set[str]) -> set[str]:
    """Технології зі словника, що трапляються в тексті окремим словом (без регістру)."""
    return {term for term in vocabulary if re.search(rf"(?<!\w){re.escape(term.lower())}(?!\w)", text.lower())}

In [ ]:
vocabulary = set(SKILL_GROUPS) | {"Celery", "Kubernetes"}
wanted, have = keywords(VACANCY, vocabulary), keywords(CV_TEXT, vocabulary)
print("у вакансії:      ", sorted(wanted))
print("є в CV:          ", sorted(wanted & have))
print("бракує в CV:     ", sorted(wanted - have))

`pytest` у CV є лише в пункті про CI («pytest on 2 Python versions»), тож він знайшовся. `Celery` і `Kubernetes` у курсі не було — і **дописувати їх не можна**: дзеркалення — про назви того, що ти справді вмієш, а не про вигадані навички. Чесна відповідь на співбесіді: «фонові задачі робив через `BackgroundTasks` і Redis (урок 40), Celery — наступний крок».

---
## 4. CV як дані → HTML

У проєкті `cv_maker` CV — готовий HTML. Якщо CV кількох версій (Backend, Data, академічне), зручніше тримати **дані** окремо й генерувати HTML. Але текст, вставлений у HTML, треба **екранувати**.

### Приклад 4.1. Знайди помилку: текст без екранування

CV вигаданої студентки як словник. У навичках — рядок `C++ & Rust <basics>`. `render_unsafe` вставляє текст у розмітку «як є».

🔮 **Прогноз:** що побачить рекрутер у цьому пункті?

In [ ]:
CV = {
    "name": "Ivanka Prykladenko",                          # вигадана студентка
    "headline": "Junior Python Backend Developer",
    "contacts": {"Email": "ivanka.prykladenko@example.com", "GitHub": "https://github.com/example-student"},
    "summary": summary(STRONG_SUMMARY)[0],
    "sections": {
        "Technical Skills": ["Backend: Python, FastAPI, Django, SQLAlchemy, Pydantic",
                             "Data & storage: PostgreSQL, Redis, Alembic",
                             "Infrastructure: Docker, Compose, nginx, GitHub Actions",
                             "Also: C++ & Rust <basics>"],
        "Projects": STRONG,
        "Final Project": ["Designed a keyword news digest bot on FastAPI and aiogram: subscriptions, hourly scraping, LLM summaries.",
                          "Optimized title search in PostgreSQL with a trigram index on lower(title): 372 ms to 0.14 ms, measured with EXPLAIN ANALYZE."],
        "Experience": ["Logistics Coordinator — Example Logistics (fictional), 2021–2025: automated weekly shipment reports with Python."],
        "Education": ["Python Developer course (PY), 2026: final project — news digest bot on FastAPI"],
        "Languages": ["Ukrainian — Native", "English — B2"],
    },
}


def render_unsafe(cv: dict) -> str:
    parts = [f"<h1>{cv['name']}</h1>"]
    for title, items in cv["sections"].items():
        parts.append(f"<h2>{title}</h2><ul>")
        parts += [f"<li>{item}</li>" for item in items]
        parts.append("</ul>")
    return "\n".join(parts)


class ListItems(HTMLParser):
    """Текст кожного <li> — так, як його покаже браузер."""

    def __init__(self) -> None:
        super().__init__()
        self.items: list[str] = []
        self.in_li = False

    def handle_starttag(self, tag, attrs):
        if tag == "li":
            self.in_li = True
            self.items.append("")

    def handle_endtag(self, tag):
        if tag == "li":
            self.in_li = False

    def handle_data(self, data):
        if self.in_li:
            self.items[-1] += data


def shown_items(page: str) -> list[str]:
    parser = ListItems()
    parser.feed(page)
    return parser.items

In [ ]:
unsafe = shown_items(render_unsafe(CV))
print("у даних:      ", CV["sections"]["Technical Skills"][3])
print("покаже браузер:", repr(unsafe[3]))

<details>
<summary>Відповідь</summary>

`<basics>` браузер сприйняв як (невідомий) тег і не показав — від пункту лишилось «Also: C++ & Rust ». Той самий клас помилки, що з ім'ям користувача в HTML-повідомленні бота (урок 48): усе, що вставляється в розмітку, — через екранування. Для CV це «лише» зіпсований текст; для сайту, куди текст пише будь-хто, — XSS.
</details>

### Приклад 4.2. `html.escape` — і текст доходить як є

Невелика зміна: кожен текст з даних — через `html.escape` (`<` → `&lt;`, `&` → `&amp;`, лапки → `&quot;`). Додамо й контакти: email — посиланням `mailto:`.

In [ ]:
import html


def render(cv: dict) -> str:
    """HTML тіла CV з даних; увесь текст — через html.escape."""
    e = html.escape
    contacts = "<br>\n".join(
        f'{e(k)}: <a href="{e(("mailto:" if k == "Email" else "") + v)}">{e(v.removeprefix("https://"))}</a>'
        for k, v in cv["contacts"].items())
    parts = [f"<h1>{e(cv['name'])}</h1>",
             f'<div class="contact">{e(cv["headline"])}<br>\n{contacts}</div>',
             f"<h2>Professional Summary</h2>\n<p>{e(cv['summary'])}</p>"]
    for title, items in cv["sections"].items():
        parts.append(f"<h2>{e(title)}</h2>\n<ul>")
        parts += [f"<li>{e(item)}</li>" for item in items]
        parts.append("</ul>")
    return "\n".join(parts)

In [ ]:
body = render(CV)
print(body.splitlines()[0], "…")
print([line for line in body.splitlines() if "Rust" in line][0])
assert shown_items(body)[3] == "Also: C++ & Rust <basics>"        # браузер покаже рівно те, що в даних
assert shown_items(body) == [item for items in CV["sections"].values() for item in items]

У розмітці — `C++ &amp; Rust &lt;basics&gt;`, а браузер (і `HTMLParser`) показує рівно `C++ & Rust <basics>`. Друга перевірка — що **всі** пункти доходять без змін.

### Приклад 4.3. Сторінка зі стилем шаблону

Задача: зробити CV, яке виглядає як `cv_template.html` (кольори, розміри, розриви сторінок), але з наших даних. Візьмемо з шаблону блок `<style>` і обгорнемо `render(CV)` у повний документ з `<meta charset="UTF-8">`. Результат — файл у `cv_output/` і перегляд прямо тут, у рамці (`iframe`), щоб стилі CV не змішалися зі стилями ноутбука.

In [ ]:
from IPython.display import HTML, display


def template_style(template_html: str) -> str:
    """Блок <style>…</style> з шаблону."""
    return re.search(r"<style>.*?</style>", template_html, re.S).group(0)


def page(cv: dict, style: str) -> str:
    return (f'<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<title>CV — {html.escape(cv["name"])}</title>\n'
            f"{style}\n</head>\n<body>\n{render(cv)}\n</body>\n</html>\n")


def preview(document: str, height: int = 520) -> None:
    """Показати HTML у рамці: srcdoc — документ усередині атрибута, тому теж екрануємо."""
    display(HTML(f'<div><iframe srcdoc="{html.escape(document)}" style="width:100%;height:{height}px;border:1px solid #ccc"></iframe></div>'))

In [ ]:
STYLE = template_style(TEMPLATE_HTML)
cv_html = page(CV, STYLE)
cv_file = OUT / "cv_ivanka_prykladenko.html"
cv_file.write_text(cv_html, encoding="utf-8")
print(f"{cv_file}: {cv_file.stat().st_size} байт, слів тексту: {len(' '.join(ats_text(cv_html)).split())}")
preview(cv_html)

Стиль — той самий CSS, що в шаблоні, тож заголовки, кольори і `break-after: avoid` для `h2` працюють і тут. Файл можна відкрити в браузері й надрукувати в PDF (`Ctrl+P` → «Зберегти як PDF») — це запасний шлях, якщо генераторів з розділу 5 немає. У Colab файл — у панелі «Файли» зліва, папка `cv_output`.

### Приклад 4.4. Дві версії з одних даних

Туторіал (розділ 11): CV адаптують під вакансію — найважливіше вгорі. Задача: Backend-версія і Data-версія з того самого словника. Невелика зміна — функція переставляє розділи і пункти за ключовими словами, а `difflib` показує, чим версії відрізняються (так само, як `git diff` між гілками CV).

In [ ]:
import copy
import difflib


def variant(cv: dict, headline: str, first_section: str, keywords_first: list[str]) -> dict:
    """Копія CV: інший заголовок, розділ first_section — першим, у ньому пункти з keywords_first — вище."""
    result = copy.deepcopy(cv)
    result["headline"] = headline
    sections = result["sections"]
    items = sections[first_section]
    items.sort(key=lambda item: not any(k in item for k in keywords_first))   # стабільне сортування
    result["sections"] = {first_section: items, **{k: v for k, v in sections.items() if k != first_section}}
    return result


BACKEND = variant(CV, "Junior Python Backend Developer", "Projects", ["FastAPI", "Docker"])
DATA = variant(CV, "Junior Python Developer (Data)", "Technical Skills", ["PostgreSQL", "Redis"])

In [ ]:
for name, cv in (("backend", BACKEND), ("data", DATA)):
    (OUT / f"cv_{name}.html").write_text(page(cv, STYLE), encoding="utf-8")
diff = difflib.unified_diff(ats_text(page(BACKEND, STYLE)), ats_text(page(DATA, STYLE)),
                            "cv_backend", "cv_data", lineterm="", n=0)
for line in list(diff)[:10]:
    print(line[:110])

Дані одні, версій дві: змінилися заголовок і порядок розділів, а жоден факт не вигадано. Тримай `cv_data.py` (дані) і генератор у Git — і кожна версія CV стає комітом, який можна порівняти й відновити.

### Задача 3. Чек-лист готового CV

Напиши `cv_checklist(html_text)` → словник `{пункт: True/False}` рівно з п'ятьма пунктами (текст сторінки бери через `ats_text`):

- `"без заготовок"` — у **тексті** сторінки немає жодного `[…]`;
- `"email"` — є посилання `mailto:`;
- `"GitHub"` — є посилання на `github.com/`;
- `"розділи"` — є заголовки `Projects` і `Technical Skills` (рядки тексту, що дорівнюють їм);
- `"обсяг"` — у тексті від 150 до 900 слів (1–2 сторінки).

Перевірка: CV Іванки проходить усе, шаблон — ні (заготовки), а сторінка без контактів — без email і GitHub.

In [ ]:
def cv_checklist(html_text: str) -> dict[str, bool]:
    # BEGIN SOLUTION
    text_lines = ats_text(html_text)
    text = " ".join(text_lines)
    return {
        "без заготовок": re.search(r"\[[^\]]+\]", text) is None,
        "email": "mailto:" in html_text,
        "GitHub": "github.com/" in html_text,
        "розділи": {"Projects", "Technical Skills"} <= set(text_lines),
        "обсяг": 150 <= len(text.split()) <= 900,
    }
    # END SOLUTION

In [ ]:
no_contacts = page({**CV, "contacts": {}}, STYLE)
assert isinstance(cv_checklist(cv_html), dict) and len(cv_checklist(cv_html)) == 5, "поверни словник рівно з п'ятьма пунктами"
for name, document in (("CV Іванки", cv_html), ("шаблон", TEMPLATE_HTML), ("без контактів", no_contacts)):
    report = cv_checklist(document)
    print(f"{name:<14}", "  ".join(("✅ " if ok else "❌ ") + item for item, ok in report.items()))
assert all(cv_checklist(cv_html).values()), "готове CV має пройти всі пункти — перевір, який False"
assert not cv_checklist(TEMPLATE_HTML)["без заготовок"], "у шаблоні є [заготовки] — пункт має бути False"
assert cv_checklist(no_contacts)["email"] is False and cv_checklist(no_contacts)["GitHub"] is False
print("✅ Задача 3 розв'язана")

Шаблон не проходить «без заготовок» — так і має бути, доки ти його не заповнив; `mailto:` у ньому теж немає (email — текстом), а тексту замало для «обсягу», бо замість змісту — підказки. Запусти `cv_checklist` на своєму HTML перед кожним надсиланням: `cv_checklist(Path("cv_моє.html").read_text(encoding="utf-8"))`.

---
## 5. PDF

Два генератори з папки проєкту:

| | `generate_cv_pdf.py` (стартовий код) | `generate_cv_pdf_chromium.py` |
|---|---|---|
| рушій | `wkhtmltopdf` (QtWebKit, проєкт архівований) | Chromium через Playwright |
| встановлення | `pip install pdfkit` + системна програма | `pip install playwright && playwright install chromium` |
| CV викладача / шаблон | 5 / 1 сторінка | 6 / 2 сторінки |

Різниця в сторінках — не помилка: `wkhtmltopdf` за замовчуванням зменшує сторінку, щоб вмістити більше (smart shrinking). З `--disable-smart-shrinking` він дає ті самі 6 і 2 сторінки, що Chromium.

### Приклад 5.1. Який генератор є тут

In [ ]:
def pick_generator() -> str | None:
    """'chromium', якщо є Playwright; 'wkhtmltopdf', якщо є pdfkit і програма; інакше None."""
    if importlib.util.find_spec("playwright"):
        return "chromium"
    if importlib.util.find_spec("pdfkit") and shutil.which("wkhtmltopdf"):
        return "wkhtmltopdf"
    return None


def pdf_pages(path: Path) -> int:
    """Кількість сторінок PDF — за об'єктами /Type /Page у файлі."""
    return len(re.findall(rb"/Type\s*/Page[^s]", path.read_bytes()))

In [ ]:
GENERATOR = pick_generator()
print("генератор PDF:", GENERATOR or "немає (Playwright і wkhtmltopdf не встановлені)")

### Приклад 5.2. PDF — або HTML, якщо генератора немає

Задача: одна клітинка, що працює всюди. Є генератор — запускаємо скрипт з папки проєкту в окремому процесі й рахуємо сторінки. Немає — пояснюємо, як встановити, і показуємо HTML (приклад 4.3 уже зберіг його у `cv_output/`). Установку Playwright у Colab не робимо автоматично: вона завантажує браузер (~150 МБ).

In [ ]:
pdf_file = OUT / "cv_ivanka_prykladenko.pdf"
script = {"chromium": "generate_cv_pdf_chromium.py", "wkhtmltopdf": "generate_cv_pdf.py"}.get(GENERATOR)
if script:
    run = subprocess.run([sys.executable, str(HERE / script), str(cv_file.resolve()), str(pdf_file.resolve())],
                         capture_output=True, text=True)
    if run.returncode == 0 and pdf_file.exists():
        print(run.stdout.strip(), f"— {pdf_pages(pdf_file)} стор., {pdf_file.stat().st_size // 1024} КБ")
    else:
        print("генератор не спрацював:", (run.stderr or run.stdout).strip().splitlines()[-1][:200])
else:
    print("PDF-генератора тут немає. Colab / Linux:")
    print("  !pip install -q playwright && playwright install --with-deps chromium")
    print("Без генератора: відкрий", cv_file, "у браузері → Ctrl+P → «Зберегти як PDF».")
    print("Вигляд сторінки — у рамці вище (приклад 4.3).")

Вивід справжнього запуску з Playwright (Chromium 1248):

```text
PDF created: …/cv_output/cv_ivanka_prykladenko.pdf — 2 стор., 28 КБ
```

Дві сторінки на ~200 слів — через великі шрифти шаблону (`h1` 34 px, `h2` 25 px) і поля 15 мм. Для CV на одну сторінку — скороти текст або зменш `font-size` у CSS, а не покладайся на стиснення.

`generate_cv_pdf_chromium.py` відкриває файл як `file://` (тому підхоплюються й локальні картинки та стилі) і друкує A4 з полями 15 мм. Якщо `playwright install` зробити не можна, а Chrome вже є, — шлях до нього в змінній `CHROMIUM_PATH`.

### Приклад 5.3. CSS для друку: заголовок не лишається сам унизу сторінки

Туторіал (розділ 8): у PDF заголовок розділу не має опинитися останнім рядком сторінки, а «шапка» проєкту — розірватися. Задача: перевірити це в CSS шаблону, не відкриваючи PDF. Невелика зміна — розберемо `<style>` на правила `селектор { властивості }` і знайдемо, які селектори мають `break-after: avoid` і `break-inside: avoid`.

In [ ]:
def css_rules(style: str) -> dict[str, str]:
    """{селектор: властивості} з блоку <style> (кілька селекторів через кому — кожен окремо)."""
    rules: dict[str, str] = {}
    for selectors, body in re.findall(r"([^{}]+)\{([^}]*)\}", style.replace("<style>", "").replace("</style>", "")):
        for selector in selectors.split(","):
            rules[selector.strip()] = rules.get(selector.strip(), "") + " ".join(body.split())
    return rules


def selectors_with(rules: dict[str, str], prop: str) -> list[str]:
    return sorted(s for s, body in rules.items() if re.search(rf"(?<![\w-]){prop}\s*:\s*avoid", body))

In [ ]:
rules = css_rules(STYLE)
print("break-after: avoid  →", selectors_with(rules, "break-after"))
print("break-inside: avoid →", selectors_with(rules, "break-inside"))
print("page-break-* (для wkhtmltopdf) теж є:", bool(selectors_with(rules, "page-break-after")))

`h2` і `h3` «прилипають» до наступного блоку, а шапки проєктів, досвіду й освіти не розриваються. Кожне правило записане двічі: `break-*` — сучасна назва (Chromium), `page-break-*` — стара (QtWebKit у `wkhtmltopdf`). Тому шаблон однаково правильно друкується обома генераторами.

### Прибираємо

Згенеровані CV — у `cv_output/`. Якщо хочеш їх залишити (відкрити в браузері, завантажити з Colab), не запускай цю клітинку.

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
print("cv_output прибрана:", not OUT.exists())

---
## Самоперевірка

1. Чим сильний пункт CV відрізняється від слабкого?
2. Чому для CV правильне число тестів — те, що друкує `pytest`, а не кількість функцій `test_*`?
3. Чому не можна дописати в CV `Celery`, якщо вакансія його просить, а ти ним не користувався?
4. Навіщо тримати дані CV окремо від HTML?
5. Що станеться з `<` і `&` у тексті без `html.escape`?
6. Чому однаковий HTML дає 5 сторінок у `wkhtmltopdf` і 6 — у Chromium?

<details>
<summary>Відповіді</summary>

1. Сильний — дієслово дії, що зроблено, чим і з яким результатом; слабкий описує процес («Worked on…»).
2. `parametrize` перетворює одну функцію на кілька тестів; `343 passed` — число, яке побачить будь-хто, хто запустить тести.
3. Дзеркалення — про назви навичок, які ти справді маєш. Вигадана навичка виявиться на першому ж технічному питанні — і зіпсує довіру до решти CV.
4. Одні дані — кілька версій під різні вакансії; версії — у Git; дизайн змінюється без переписування змісту.
5. `<…>` браузер сприйме як тег і сховає, `&` може стати початком сутності; на сайтах це XSS.
6. Smart shrinking у `wkhtmltopdf` зменшує вміст; з `--disable-smart-shrinking` сторінок стільки ж.
</details>

## Далі

- Книга курсу: [Бонус. CV розробника](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/bonus_cv/) — генератори, шаблон, туторіал «сильне CV» ([довідник](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/cv/cv_tutorial/)).
- У папці проєкту: `cv_template.html` → `python generate_cv_pdf_chromium.py cv_template.html CV_Name_Surname.pdf` (або `generate_cv_pdf.py` з `wkhtmltopdf`).
- Капстоун (уроки 52–53): фінальний проєкт — найсильніший пункт твого CV, а факти для нього ти вже зібрав для питчу.